In [0]:
%sql
create catalog if not exists dqx_sandbox;
create schema if not exists dqx_sandbox.dqx_bronze;

### Input Table - customer

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date

data = [
    (1, "Alice Smith", date(1990, 5, 14), "alice.smith@email.com", "123-456-7890", "New York", "NY", "USA", 1000.50, "Active"),
    (2, "Bob Johnson", date(1985, 8, 23), "bob.johnson@email.com", "234-567-8901", "Los Angeles", "CA", "USA", 850.75, "Active"),
    (3, "Charlie Lee", date(1992, 12, 2), "charlie$lee@email.com", "127-456-7890", "Chicago", "IL", "USA", 1200.00, "Active"),
    (4, "Diana King", date(1988, 3, 17), "diana.king@email.com", "456-789-0123", "Houston", "TX", "USA", -10.25, "Active"),
    (5, "Ethan Brown", date(1995, 7, 30), "ethan.brown@email.com", "567-890-1234", "Phoenix", "AZ", "USA", 700.00, "Active"),
    (None, "Fiona Clark", date(1991, 11, 5), "fiona.clark@email.com", "456-789-0123", "Philadelphia", "PA", "USA", 1100.10, "Active"),
    (None, "George Hall", date(1983, 6, 21), "george.hall@email.com", "789-012-3456", "San Antonio", "TX", "USA", 600.00, "Active"),
    (8, "Hannah Adams", date(1994, 9, 12), "hannah.adams@email.com", "890-123-4567", "San Diego", "CA", "USA", 1300.00, "Active"),
    (9, "Ian Wright", date(1987, 2, 8), "ian.wright@email.com", "901-234-5678", "Dallas", "TX", "USA", 750.50, "Active"),
    (10, "Julia Turner", date(1993, 4, 25), "julia.turner@email.com", "012-345-6789", "San Jose", "CA", "USA", 900.00, "Active")
]

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), False),
    StructField("customer_dob", DateType(), False),
    StructField("customer_email", StringType(), False),
    StructField("customer_phone", StringType(), False),
    StructField("customer_city", StringType(), False),
    StructField("customer_state", StringType(), False),
    StructField("customer_country", StringType(), False),
    StructField("customer_account_balance", DoubleType(), False),
    StructField("customer_status", StringType(), False)
])

customer_sample_df = spark.createDataFrame(data, schema)
spark.sql("DROP TABLE IF EXISTS dqx_sandbox.dqx_bronze.customer")
customer_sample_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dqx_sandbox.dqx_bronze.customer")
display(table('dqx_sandbox.dqx_bronze.customer'))


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType
from datetime import date

# Sample order data referencing customer_id as foreign key, now with product_id
order_data = [
    (101, 1, date(2026, 2, 20), 1001, 1200.00),
    (102, 2, date(2026, 2, 18), 1002, 800.00),
    (103, 2, date(2026, 2, 15), 1003, 950.00),
    (104, 6, date(2026, 2, 10), 1003, 150.00),
    (105, 8, date(2026, 2, 5), 1005, 300.00),
    (106, 8, date(2026, 2, 1), 1001, 75.00)
]

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),  # Foreign key to customer table
    StructField("order_date", DateType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("order_amount", DoubleType(), False)
])

order_sample_df = spark.createDataFrame(order_data, order_schema)
spark.sql("DROP TABLE IF EXISTS dqx_sandbox.dqx_bronze.order")
order_sample_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dqx_sandbox.dqx_bronze.order")
display(table('dqx_sandbox.dqx_bronze.order'))

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Product table
product_data = [
    (1001, "Laptop", 1500.00),
    (1002, "Tablet", 900.00),
    (1003, "Smartphone", 1000.00),
    (1005, "Monitor", 350.00)
]

product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("product_price", DoubleType(), False)
])

product_sample_df = spark.createDataFrame(product_data, product_schema)
spark.sql("DROP TABLE IF EXISTS dqx_sandbox.dqx_bronze.product")
product_sample_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dqx_sandbox.dqx_bronze.product")
display(table('dqx_sandbox.dqx_bronze.product'))

In [0]:
# Payment table
payment_data = [
    (201, 101, "Credit Card", 1200.00),
    (202, 102, "PayPal", 800.00),
    (203, 103, "Credit Card", 950.00),
    (204, 104, "Bank Transfer", 150.00),
    (205, 105, "Credit Card", 300.00),
    (206, 106, "PayPal", 75.00)
]

payment_schema = StructType([
    StructField("payment_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("payment_method", StringType(), False),
    StructField("payment_amount", DoubleType(), False)
])

payment_sample_df = spark.createDataFrame(payment_data, payment_schema)
spark.sql("DROP TABLE IF EXISTS dqx_sandbox.dqx_bronze.payment")
payment_sample_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dqx_sandbox.dqx_bronze.payment")
display(table('dqx_sandbox.dqx_bronze.payment'))

In [0]:
tbl_schema = customer_sample_df.schema
col_types = [f"{field.name} {field.dataType.simpleString()}" for field in tbl_schema]
col_types_str = " , ".join(col_types)
col_types_str